In [3]:
import os
import io
import gzip
import json
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.patches as mpatches

from dotenv import load_dotenv
from google.cloud import storage

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

In [4]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

GCS_PROJECT = "plm-study-484223"
GCS_BUCKET = "domainome-data"
prefix = "ESM2/ft_per_pfam/global_finetune/"

client = storage.Client(project=GCS_PROJECT)
bucket = client.bucket(GCS_BUCKET)


In [6]:
# make metadata dataframe first with best config
# best_config, n_domains

rows = []

blobs = bucket.list_blobs(prefix=prefix)

for blob in blobs:
    
    row = {}

    if blob.name.endswith("metadata.json"):
        
        parts = blob.name.split("/")
        pfam = parts[-2]
        row["model"] = pfam

        dmeta = json.loads(blob.download_as_text())
        row["n_domains_model"] = dmeta["n_domains"] 

        best_config = dmeta["best_config"]
        row['lr'] = best_config['lr']
        row['lora_r'] = best_config['lora_r']

        rows.append(row)

df_train = pd.DataFrame(rows)



In [7]:
df_train.head()

,model,n_domains_model,lr,lora_r
0,PF00010,2,0.0010,128
1,PF00013,9,0.0050,16
2,PF00018,36,0.0001,16
3,PF00030,12,0.0050,8
4,PF00035,2,0.0010,16


In [9]:
# add data from sweep_curves for each pfam and best_config
# epochs_ran, best_val_loss, best_val_rho, train_sampled_domains with count

rows = []

blobs = bucket.list_blobs(prefix=prefix)

for blob in blobs:
    
    row = {}

    if blob.name.endswith("sweep_curves.json"):
        
        parts = blob.name.split("/")
        pfam = parts[-2]
        row["model"] = pfam

        dtrain = json.loads(blob.download_as_text())

        list_best_train = [
                                    d for d in dtrain
                                    if d.get("lr") == best_config['lr']
                                    and d.get("lora_r") == best_config['lora_r']
                                ]
        
        best_train = list_best_train[0]

        row["epochs_ran"] = best_train["epochs_ran"] 

        train_doms = best_train['train_sampled_domains']
        train_doms = pd.DataFrame({'domain_id': sum(train_doms, [])})
        train_doms = train_doms.value_counts().reset_index(name='count')
        row["train_doms"] = train_doms.set_index('domain_id').to_dict()['count']

        val_doms = best_train['val_sampled_domains']
        train_doms = pd.DataFrame({'domain_id': sum(val_doms, [])})
        train_doms = train_doms.value_counts().reset_index(name='count')
        row["val_doms"] = train_doms.set_index('domain_id').to_dict()['count']
        

        rows.append(row)

df_train_more = pd.DataFrame(rows)

In [10]:
df_train_more.head()

,model,epochs_ran,train_doms,val_doms
0,PF00010,9,"{'P50539_PF00010_70': 94, 'Q05195_PF00010_58':...","{'P50539_PF00010_70': 90, 'Q05195_PF00010_58':..."
1,PF00013,4,"{'Q86XN8_PF00013_273': 13, 'Q06787_PF00013_217...","{'Q92945_PF00013_142': 14, 'Q06787_PF00013_217..."
2,PF00018,5,"{'O43295_PF00018_745': 9, 'P06241_PF00018_83':...","{'Q15811_PF00018_1156': 7, 'P08631_PF00018_80'..."
3,PF00030,3,"{'P07315_PF00030_4': 9, 'P07320_PF00030_90': 9...","{'P05813_PF00030_31': 8, 'Q8N1P7_PF00030_1358'..."
4,PF00035,4,"{'O95793_PF00035_183': 51, 'P78563_PF00035_80'...","{'P78563_PF00035_80': 43, 'O95793_PF00035_183'..."


In [11]:
df_merged = df_train.merge(df_train_more, on="model")

In [12]:
df_merged.head()

,model,n_domains_model,lr,lora_r,epochs_ran,train_doms,val_doms
0,PF00010,2,0.0010,128,9,"{'P50539_PF00010_70': 94, 'Q05195_PF00010_58':...","{'P50539_PF00010_70': 90, 'Q05195_PF00010_58':..."
1,PF00013,9,0.0050,16,4,"{'Q86XN8_PF00013_273': 13, 'Q06787_PF00013_217...","{'Q92945_PF00013_142': 14, 'Q06787_PF00013_217..."
2,PF00018,36,0.0001,16,5,"{'O43295_PF00018_745': 9, 'P06241_PF00018_83':...","{'Q15811_PF00018_1156': 7, 'P08631_PF00018_80'..."
3,PF00030,12,0.0050,8,3,"{'P07315_PF00030_4': 9, 'P07320_PF00030_90': 9...","{'P05813_PF00030_31': 8, 'Q8N1P7_PF00030_1358'..."
4,PF00035,2,0.0010,16,4,"{'O95793_PF00035_183': 51, 'P78563_PF00035_80'...","{'P78563_PF00035_80': 43, 'O95793_PF00035_183'..."


In [13]:
df_merged['train_doms'].iloc[4]

{'O95793_PF00035_183': 51, 'P78563_PF00035_80': 29}

In [14]:
df_merged.to_csv("/Users/johnhutchens/Desktop/Practicum/Data/ESM2_pfam_metadata.csv.gz", index=False, compression='gzip')

## Bad PFAM models
### PF00046

In [85]:
blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00046/sweep_curves.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()

train_PF00046 = json.loads(data)
blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00046/metadata.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()

meta_PF00046 = json.loads(data)
train_dict = train_PF00046
best_config = meta_PF00046["best_config"]

list_best_train_PF00046 = [
    d for d in train_dict
    if d.get("lr") == best_config['lr']
    and d.get("lora_r") == best_config['lora_r']
]

print(f"Number of best lora configs: {len(list_best_train_PF00046)}")
best_train_PF00046 = list_best_train_PF00046[0]
# best_train_PF00046
doms = best_train_PF00046['train_sampled_domains']
PF00046_train_doms = pd.DataFrame({'domain_id': sum(doms, [])})
PF00046_train_doms = PF00046_train_doms.value_counts().reset_index(name='count')

Number of best lora configs: 1


In [86]:
PF00046_train_doms

,domain_id,count
0,O75360_PF00046_71,5
1,P35548_PF00046_144,4
2,P26367_PF00046_213,4
3,O95076_PF00046_155,4
4,Q8NFW5_PF00046_73,3
5,P32243_PF00046_40,3
6,O15266_PF00046_119,3
7,P20265_PF00046_356,3
8,O95475_PF00046_134,3
9,P23759_PF00046_219,2


### PF00536

In [88]:
blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00536/sweep_curves.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
train_PF00536 = json.loads(data)

blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00536/metadata.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
meta_PF00536 = json.loads(data)

train_dict = train_PF00536
best_config = meta_PF00536["best_config"]

list_best_train_PF00536 = [
    d for d in train_dict
    if d.get("lr") == best_config['lr']
    and d.get("lora_r") == best_config['lora_r']
]

print(f"Number of best lora configs: {len(list_best_train_PF00536)}")
best_train_PF00536 = list_best_train_PF00536[0]

doms = best_train_PF00536['train_sampled_domains']
PF00536_train_doms = pd.DataFrame({'domain_id': sum(doms, [])})
PF00536_train_doms = PF00536_train_doms.value_counts().reset_index(name='count')

Number of best lora configs: 1


In [89]:
PF00536_train_doms

,domain_id,count
0,Q9NYL2_PF00536_330,24
1,O94885_PF00536_630,20
2,O94830_PF00536_389,16
3,Q8WXI2_PF00536_9,16
4,Q68DC2_PF00536_774,12
5,Q7Z6G8_PF00536_812,12


## Good PFAM models
### PF00505

In [90]:
blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00505/sweep_curves.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
train_PF00505 = json.loads(data)

blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00505/metadata.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
meta_PF00505 = json.loads(data)

train_dict = train_PF00505
best_config = meta_PF00505["best_config"]

list_best_train_PF00505 = [
    d for d in train_dict
    if d.get("lr") == best_config['lr']
    and d.get("lora_r") == best_config['lora_r']
]

print(f"Number of best lora configs: {len(list_best_train_PF00505)}")
best_train_PF00505 = list_best_train_PF00505[0]

doms = best_train_PF00505['train_sampled_domains']
PF00505_train_doms = pd.DataFrame({'domain_id': sum(doms, [])})
PF00505_train_doms = PF00505_train_doms.value_counts().reset_index(name='count')

Number of best lora configs: 1


In [91]:
PF00505_train_doms

,domain_id,count
0,P35712_PF00505_622,21
1,O94993_PF00505_338,19
2,P35711_PF00505_557,16
3,Q06945_PF00505_60,15
4,P35716_PF00505_50,15
5,Q86U86_PF00505_1383,15
6,P41225_PF00505_140,14
7,O15405_PF00505_256,13
8,Q9H6I2_PF00505_69,13
9,P17480_PF00505_408,12


### PF00013

In [92]:
blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00013/sweep_curves.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
train_PF00013 = json.loads(data)

blob_path = 'ESM2/ft_per_pfam/global_finetune/PF00013/metadata.json'
blob = bucket.blob(blob_path)
data = blob.download_as_text()
meta_PF00013 = json.loads(data)

train_dict = train_PF00013
best_config = meta_PF00013["best_config"]

list_best_train_PF00013 = [
    d for d in train_dict
    if d.get("lr") == best_config['lr']
    and d.get("lora_r") == best_config['lora_r']
]

print(f"Number of best lora configs: {len(list_best_train_PF00013)}")
best_train_PF00013 = list_best_train_PF00013[0]

doms = best_train_PF00013['train_sampled_domains']
PF00013_train_doms = pd.DataFrame({'domain_id': sum(doms, [])})
PF00013_train_doms = PF00013_train_doms.value_counts().reset_index(name='count')

Number of best lora configs: 1


In [93]:
PF00013_train_doms

,domain_id,count
0,Q86XN8_PF00013_273,13
1,Q06787_PF00013_217,12
2,Q5U5Q3_PF00013_325,11
3,Q96I24_PF00013_75,11
4,P61978_PF00013_385,8
5,Q96AE4_PF00013_98,8
6,Q15365_PF00013_278,7
7,Q92945_PF00013_142,5
8,Q96AE4_PF00013_184,5
